In [4]:
!pip install symspellpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 14.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [symspellpy]

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
from __future__ import annotations

from random import Random
from unicodedata import normalize

from symspellpy import SymSpell, Verbosity

RNG = Random(123)

HEALTH_BASE = [
    'hastane', 'doktor', 'hekim', 'hemşire', 'ilaç', 'reçete', 'muayene',
    'randevu', 'tedavi', 'ameliyat', 'enfeksiyon', 'aşı', 'ateş', 'öksürük',
    'boğaz', 'solunum', 'kalp', 'damar', 'beyin', 'sinir', 'cilt', 'kulak',
    'burun', 'göz', 'diş', 'diyabet', 'tansiyon', 'obezite', 'kanser',
    'nefroloji', 'kardiyoloji', 'nöroloji', 'ortopedi', 'psikiyatri', 'acil',
    'yoğunbakım', 'laboratuvar', 'rapor', 'tahlil', 'semptom', 'teşhis'
]

HEALTH_COMPOUNDS = [
    'acilservis', 'ağrıkesici', 'kanbasıncı', 'kalpkrizi', 'beyincerrahisi',
    'gözmuayenesi', 'kulakburunboğaz', 'aşılama', 'ilaçtakibi', 'randevusistemi',
    'hastakaydı', 'tahlilsonucu', 'tedaviplanı', 'solunumyolu', 'doktorraporu',
    'hemşirelik', 'ameliyatöncesi', 'ameliyatsonrası', 'kanşekeri', 'nabızölçer'
]

HEALTH_SPLIT_PARTS = [
    'servis', 'ağrı', 'kesici', 'kan', 'basıncı', 'kalp', 'krizi', 'beyin', 'cerrahisi',
    'göz', 'muayenesi', 'kulak', 'burun', 'boğaz', 'aşı', 'lama', 'ilaç', 'takibi',
    'randevu', 'sistemi', 'hasta', 'kaydı', 'tahlil', 'sonucu', 'tedavi', 'planı',
    'solunum', 'yolu', 'doktor', 'raporu', 'ameliyat', 'öncesi', 'sonrası', 'kan',
    'şekeri', 'nabız', 'ölçer', 'bakım', 'aşılama'
]

DICTIONARY_WORDS = sorted(set(HEALTH_BASE + HEALTH_COMPOUNDS + HEALTH_SPLIT_PARTS))

max_edit_distance = 2
prefix_length = 7
sym_spell = SymSpell(max_dictionary_edit_distance=max_edit_distance, prefix_length=prefix_length)
split_spell = SymSpell(max_dictionary_edit_distance=max_edit_distance, prefix_length=prefix_length)


for term in DICTIONARY_WORDS:
    if term in HEALTH_BASE:
        freq = 20
    elif term in HEALTH_COMPOUNDS:
        freq = 10
    else:
        freq = 5
    sym_spell.create_dictionary_entry(term, freq)
    if term in HEALTH_BASE or term in HEALTH_SPLIT_PARTS:
        split_spell.create_dictionary_entry(term, freq)


def remove_diacritics(text: str) -> str:
    return normalize('NFKD', text).encode('ascii', 'ignore').decode('ascii')


def typo_transform(word: str, forced_type: str | None = None) -> tuple[str, str]:
    typo_type = forced_type or RNG.choice(['diacritic', 'transpose', 'delete', 'replace', 'double'])

    if typo_type == 'diacritic':
        return remove_diacritics(word), typo_type

    if typo_type == 'transpose' and len(word) > 2:
        i = RNG.randrange(len(word) - 1)
        letters = list(word)
        letters[i], letters[i + 1] = letters[i + 1], letters[i]
        return ''.join(letters), typo_type

    if typo_type == 'delete' and len(word) > 2:
        i = RNG.randrange(len(word))
        return word[:i] + word[i + 1:], typo_type

    if typo_type == 'replace' and len(word) > 0:
        i = RNG.randrange(len(word))
        replacement = 'a' if word[i] != 'a' else 'e'
        return word[:i] + replacement + word[i + 1:], typo_type

    if typo_type == 'double' and len(word) > 0:
        i = RNG.randrange(len(word))
        return word[:i] + word[i] + word[i:], typo_type

    return word, 'none'


def correct_single_word(typo: str) -> tuple[str | None, int]:
    suggestions = sym_spell.lookup(typo, Verbosity.CLOSEST, max_edit_distance)
    if not suggestions:
        return None, -1
    return suggestions[0].term, suggestions[0].distance


def correct_split_phrase(typo_phrase: str) -> tuple[str | None, int]:
    suggestions = split_spell.lookup_compound(typo_phrase, max_edit_distance)
    if not suggestions:
        return None, -1
    return suggestions[0].term, suggestions[0].distance


SINGLE_WORD_EXAMPLES = [
    'hastane', 'doktor', 'hemşire', 'ilaç', 'reçete', 'muayene', 'tedavi',
    'ameliyat', 'enfeksiyon', 'aşı', 'ateş', 'öksürük', 'kalp', 'beyin'
]

COMPOUND_EXAMPLES = [
    ('acilservis', 'acilserviz'),
    ('ağrıkesici', 'agrikesici'),
    ('kanbasıncı', 'kanbasınc'),
    ('kalpkrizi', 'kalpkrizii'),
    ('beyincerrahisi', 'beyincerrahis'),
    ('gözmuayenesi', 'gozmuayenesi'),
    ('kulakburunboğaz', 'kulakburunbogaz'),
    ('ilaçtakibi', 'ilactakibi'),
    ('randevusistemi', 'randevusistem'),
    ('hastakaydı', 'hastakayyı'),
    ('tahlilsonucu', 'tahlilsonuu'),
    ('doktorraporu', 'doktorraprpu'),
    ('solunumyolu', 'solunumyoluu'),
    ('kanşekeri', 'kansekeri'),
    ('nabızölçer', 'nabızölçre'),
]

SPLIT_PHRASE_EXAMPLES = [
    ('acil servis', 'acilservis'),
    ('ağrı kesici', 'ağrikesici'),
    ('kan basıncı', 'kanbasinci'),
    ('kalp krizi', 'kalpkrizi'),
    ('beyin cerrahisi', 'beyincerrahisi'),
    ('göz muayenesi', 'gozmuayenesi'),
    ('ilaç takibi', 'ilactakibi'),
    ('randevu sistemi', 'randevusistemi'),
    ('hasta kaydı', 'hastakaydi'),
    ('tahlil sonucu', 'tahlilsonucu'),
    ('doktor raporu', 'doktorraporu'),
    ('solunum yolu', 'solunumyolu'),
    ('kan şekeri', 'kansekeri'),
    ('nabız ölçer', 'nabızolcer'),
    ('ameliyat öncesi', 'ameliyatoncesi'),
    ('ameliyat sonrası', 'ameliyatsonrasi'),
    ('aşılama planı', 'aşilamaplani'),
    ('laboratuvar raporu', 'laboratuvarraporu'),
    ('psikiyatri randevu', 'psikiyatrirandevu'),
    ('ortopedi raporu', 'ortopediraporu'),
    ('nefroloji muayenesi', 'nefrolojimuayenesi'),
    ('kardiyoloji randevu', 'kardiyolojirandevu'),
    ('teşhis raporu', 'teshisraporu'),
]

print(f'Synthetic health dictionary size: {len(DICTIONARY_WORDS)}')
print(f'Single-word examples: {len(SINGLE_WORD_EXAMPLES)}')
print(f'Compound-word examples: {len(COMPOUND_EXAMPLES)}')
print(f'Split-phrase examples: {len(SPLIT_PHRASE_EXAMPLES)}')
print('-' * 100)

print('TEKIL KELIME DUZELTME ORNEKLERI')
print('-' * 100)
for gold in SINGLE_WORD_EXAMPLES:
    typo, typo_type = typo_transform(gold)
    suggestion, distance = correct_single_word(typo)
    status = 'OK' if suggestion == gold else 'FAIL'
    print(f'{gold:20s} | typo={typo:20s} | sug={str(suggestion):20s} | d={distance:2d} | {typo_type:10s} | {status}')

print('-' * 100)
print('HATALI BIRLESIK KELIME ORNEKLERI')
print('-' * 100)
for gold, typo in COMPOUND_EXAMPLES:
    suggestion, distance = correct_single_word(typo)
    status = 'OK' if suggestion == gold else 'FAIL'
    print(f'{gold:20s} | typo={typo:20s} | sug={str(suggestion):20s} | d={distance:2d} | {status}')

print('-' * 100)
print('NORMALDE AYRI YAZILMASI GEREKEN HATALI ORNEKLER')
print('-' * 100)
for gold_phrase, typo_phrase in SPLIT_PHRASE_EXAMPLES:
    suggestion, distance = correct_split_phrase(typo_phrase)
    status = 'OK' if suggestion == gold_phrase else 'FAIL'
    print(f'{gold_phrase:20s} | typo={typo_phrase:20s} | sug={str(suggestion):20s} | d={distance:2d} | {status}')

print('-' * 100)


Synthetic health dictionary size: 84
Single-word examples: 14
Compound-word examples: 15
Split-phrase examples: 23
----------------------------------------------------------------------------------------------------
TEKIL KELIME DUZELTME ORNEKLERI
----------------------------------------------------------------------------------------------------
hastane              | typo=hastane              | sug=hastane              | d= 0 | diacritic  | OK
doktor               | typo=oktor                | sug=doktor               | d= 1 | delete     | OK
hemşire              | typo=heaşire              | sug=hemşire              | d= 1 | replace    | OK
ilaç                 | typo=ilac                 | sug=ilaç                 | d= 1 | diacritic  | OK
reçete               | typo=recete               | sug=reçete               | d= 1 | diacritic  | OK
muayene              | typo=muayane              | sug=muayene              | d= 1 | replace    | OK
tedavi               | typo=teddavi          

In [ ]:
from collections import defaultdict
from pathlib import Path
from unicodedata import normalize

import math
import re
import pandas as pd
from symspellpy import SymSpell, Verbosity

base_dir = Path("/Users/burak.yilmaz/tubitak_2247c/words")

freq_path = base_dir / "kelime_frekanslari.csv"
missing_path = base_dir / "txtde_olmayan_kelimeler_Kontrol.csv"
abbreviation_path = base_dir / "TUM_KISALTMALAR_v3.txt"
attention_path = base_dir / "dikkat edilecekler_v2.txt"

EDIT_DISTANCES = [2, 3]
LAMBDA_VALUES = [1.0, 1.5, 2.0, 2.5, 3.0]
MAX_EDIT_DISTANCE = max(EDIT_DISTANCES)
PREFIX_LENGTH = 5
MIN_WORD_FREQUENCY = 2
TOP_SUGGESTIONS = None


def normalize_text(text: str) -> str:
    text = str(text).strip().lower()
    text = normalize("NFKC", text)
    text = re.sub(r"[^\w\sçğıöşü]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def normalize_abbreviation_key(text: str) -> str:
    text = str(text).strip().lower()
    text = normalize("NFKC", text)
    text = re.sub(r"\s+", "", text)
    return text


def remove_diacritics(text: str) -> str:
    tr_map = str.maketrans({"ç": "c", "ğ": "g", "ı": "i", "ö": "o", "ş": "s", "ü": "u"})
    text = str(text).translate(tr_map)
    text = text.replace("i̇", "i")
    return text


def safe_read_table(path: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(path, encoding="utf-8-sig")
    except Exception:
        try:
            return pd.read_excel(path)
        except Exception:
            for enc in ["utf-8", "latin1", "cp1254"]:
                try:
                    return pd.read_csv(path, encoding=enc)
                except Exception:
                    continue
            raise RuntimeError(f"Dosya okunamadı: {path}")


def load_abbreviation_rules(*paths: Path) -> dict[str, list[dict]]:
    rules: dict[str, list[dict]] = defaultdict(list)
    order_index = 0

    for source_priority, path in enumerate(paths):
        try:
            lines = path.read_text(encoding="utf-8-sig").splitlines()
        except Exception:
            lines = path.read_text(encoding="latin1").splitlines()

        for raw_line in lines:
            line = raw_line.strip()
            if not line or line.startswith("=") or line.startswith("-"):
                continue

            parts = line.split(None, 1)
            if len(parts) < 2:
                continue

            abbreviation = parts[0].strip()
            expansion = parts[1].strip()
            expansion = re.sub(r"\s*\(\*\)\s*$", "", expansion).strip()
            if not abbreviation or not expansion:
                continue

            preferred = "(*)" in raw_line
            entry = {
                "abbreviation": abbreviation,
                "expansion": expansion,
                "preferred": preferred,
                "source_priority": source_priority,
                "order_index": order_index,
            }

            for key in {
                normalize_abbreviation_key(abbreviation),
                remove_diacritics(normalize_abbreviation_key(abbreviation)),
            }:
                if key:
                    rules[key].append(entry)

            order_index += 1

    return rules


def pick_abbreviation_match(candidates: list[dict]) -> dict | None:
    if not candidates:
        return None

    ranked = sorted(
        candidates,
        key=lambda item: (
            -int(item["preferred"]),
            item["source_priority"],
            item["order_index"],
            len(item["expansion"]),
            item["expansion"],
        ),
    )
    return ranked[0]


def build_dictionary(freq_df: pd.DataFrame) -> dict[str, int]:
    dictionary_terms: dict[str, int] = {}
    for _, row in freq_df.iterrows():
        word = row["kelime"]
        freq = int(row["frekans"])

        dictionary_terms[word] = max(dictionary_terms.get(word, 0), freq)

        ascii_word = remove_diacritics(word)
        if ascii_word and ascii_word != word:
            dictionary_terms[ascii_word] = max(dictionary_terms.get(ascii_word, 0), freq)

    return dictionary_terms


def build_symspell(dictionary_terms: dict[str, int], max_edit_distance: int) -> SymSpell:
    spell = SymSpell(max_dictionary_edit_distance=max_edit_distance, prefix_length=PREFIX_LENGTH)
    for term, freq in dictionary_terms.items():
        spell.create_dictionary_entry(term, freq)
    return spell


def build_candidate_queries(text: str, spell: SymSpell) -> list[str]:
    queries = []
    for candidate in [text, remove_diacritics(text)]:
        candidate = normalize_text(candidate)
        if candidate and candidate not in queries:
            queries.append(candidate)

    segmented = spell.word_segmentation(text).corrected_string
    segmented = normalize_text(segmented)
    if segmented and segmented not in queries:
        queries.append(segmented)

    segmented_ascii = normalize_text(remove_diacritics(segmented))
    if segmented_ascii and segmented_ascii not in queries:
        queries.append(segmented_ascii)

    return queries


def add_suggestion(store: dict[str, dict], suggestion, source: str) -> None:
    term = str(getattr(suggestion, "term", "")).strip()
    if not term:
        return

    count = int(getattr(suggestion, "count", 1) or 1)
    distance = int(getattr(suggestion, "distance", len(term)) or len(term))
    existing = store.get(term)

    if existing is None:
        store[term] = {
            "term": term,
            "count": count,
            "distance": distance,
            "sources": {source},
        }
        return

    existing["count"] = max(existing["count"], count)
    existing["distance"] = min(existing["distance"], distance)
    existing["sources"].add(source)


def score_candidate(candidate: dict, lambda_value: float) -> float:
    freq = max(int(candidate.get("count", 1)), 1)
    distance = int(candidate.get("distance", 999))
    return math.log(freq) - lambda_value * distance


def rank_candidates(candidates: list[dict], lambda_value: float) -> list[dict]:
    ranked = []
    for candidate in candidates:
        scored_candidate = dict(candidate)
        scored_candidate["score"] = score_candidate(candidate, lambda_value)
        ranked.append(scored_candidate)

    ranked.sort(
        key=lambda item: (
            -item["score"],
            item["distance"],
            -item["count"],
            item["term"],
        )
    )
    return ranked


def format_candidate(candidate: dict) -> str:
    return (
        f"{candidate['term']} "
        f"(Skor: {candidate['score']:.2f}, Frekans: {candidate['count']}, Mesafe: {candidate['distance']})"
    )


def combo_suffix(edit_distance: int, lambda_value: float) -> str:
    lambda_text = str(lambda_value).replace(".", "_")
    if lambda_text.endswith("_0"):
        lambda_text = lambda_text[:-2]
    return f"ed{edit_distance}_l{lambda_text}"


def collect_candidate_store(text: str, spell: SymSpell, max_edit_distance: int) -> tuple[str, dict[str, dict]]:
    original_text = "" if pd.isna(text) else str(text)
    normalized_original = normalize_text(original_text)
    if not normalized_original:
        return normalized_original, {}

    candidate_store: dict[str, dict] = {}
    for query in build_candidate_queries(normalized_original, spell):
        for suggestion in spell.lookup(
            query,
            Verbosity.ALL,
            max_edit_distance=max_edit_distance,
            include_unknown=True,
        ):
            add_suggestion(candidate_store, suggestion, "lookup")

        for suggestion in spell.lookup_compound(query, max_edit_distance=max_edit_distance):
            add_suggestion(candidate_store, suggestion, "compound")

    return normalized_original, candidate_store


def build_result(normalized_original: str, candidate_store: dict[str, dict], lambda_value: float) -> dict:
    if not normalized_original:
        return {
            "corrected": "",
            "confidence": 0,
            "changed": False,
            "best_score": 0.0,
            "best_distance": 0,
            "best_frequency": 0,
            "all_suggestions": "",
        }

    candidates = list(candidate_store.values())
    if not candidates:
        return {
            "corrected": normalized_original,
            "confidence": 0,
            "changed": False,
            "best_score": 0.0,
            "best_distance": 0,
            "best_frequency": 0,
            "all_suggestions": "",
        }

    ranked_candidates = rank_candidates(candidates, lambda_value=lambda_value)
    best_candidate = ranked_candidates[0]
    corrected = best_candidate["term"]
    confidence = max(0, min(100, int(round(100 - (best_candidate["distance"] * 25)))))

    if TOP_SUGGESTIONS is None:
        suggestion_text = " | ".join(format_candidate(candidate) for candidate in ranked_candidates)
    else:
        suggestion_text = " | ".join(format_candidate(candidate) for candidate in ranked_candidates[:TOP_SUGGESTIONS])

    return {
        "corrected": corrected,
        "confidence": confidence,
        "changed": corrected != normalized_original,
        "best_score": round(best_candidate["score"], 4),
        "best_distance": best_candidate["distance"],
        "best_frequency": best_candidate["count"],
        "all_suggestions": suggestion_text,
    }


freq_df = safe_read_table(freq_path)
freq_df.columns = [c.strip() for c in freq_df.columns]

cols_lower = {c.lower(): c for c in freq_df.columns}
if "kelime" in cols_lower and "frekans" in cols_lower:
    if cols_lower["kelime"] != "kelime":
        freq_df = freq_df.rename(columns={cols_lower["kelime"]: "kelime"})
    if cols_lower["frekans"] != "frekans":
        freq_df = freq_df.rename(columns={cols_lower["frekans"]: "frekans"})
else:
    raise ValueError("kelime_frekanslari dosyasında 'kelime' ve 'frekans' sütunları bulunmalı.")

freq_df["kelime"] = freq_df["kelime"].astype(str).map(normalize_text)
freq_df["frekans"] = pd.to_numeric(freq_df["frekans"], errors="coerce").fillna(0).astype(int)
freq_df = freq_df[freq_df["kelime"] != ""]
freq_df = freq_df[freq_df["frekans"] >= MIN_WORD_FREQUENCY]
freq_df = freq_df.groupby("kelime", as_index=False)["frekans"].max()

print(f"Toplam sözlük boyutu: {len(freq_df):,}")

dictionary_terms = build_dictionary(freq_df)
print(f"Ham sözlük terimi sayısı: {len(dictionary_terms):,}")

missing_df = safe_read_table(missing_path)
missing_df.columns = [c.strip() for c in missing_df.columns]

source_column = "TXT_DE_OLMAYAN_KELIMELER"
if source_column not in missing_df.columns:
    raise ValueError(f"{source_column} sütunu bulunamadı.")

base_df = missing_df.copy()
base_columns = list(base_df.columns)

spell_cache = {edit_distance: build_symspell(dictionary_terms, edit_distance) for edit_distance in EDIT_DISTANCES}

candidate_cache: dict[int, list[tuple[str, dict[str, dict]]]] = {}
for edit_distance, spell in spell_cache.items():
    rows = []
    for text in base_df[source_column]:
        normalized_original, candidate_store = collect_candidate_store(
            text,
            spell=spell,
            max_edit_distance=edit_distance,
        )
        rows.append((normalized_original, candidate_store))
    candidate_cache[edit_distance] = rows

final_df = base_df.copy()

for edit_distance in EDIT_DISTANCES:
    for lambda_value in LAMBDA_VALUES:
        suffix = combo_suffix(edit_distance, lambda_value)
        corrected_values = []
        confidence_values = []
        changed_values = []
        best_score_values = []
        best_distance_values = []
        best_frequency_values = []
        suggestion_values = []

        for normalized_original, candidate_store in candidate_cache[edit_distance]:
            result = build_result(normalized_original, candidate_store, lambda_value)
            corrected_values.append(result["corrected"])
            confidence_values.append(result["confidence"])
            changed_values.append(result["changed"])
            best_score_values.append(result["best_score"])
            best_distance_values.append(result["best_distance"])
            best_frequency_values.append(result["best_frequency"])
            suggestion_values.append(result["all_suggestions"])

        final_df[f"corrected_{suffix}"] = corrected_values
        final_df[f"confidence_{suffix}"] = confidence_values
        final_df[f"changed_{suffix}"] = changed_values
        final_df[f"best_score_{suffix}"] = best_score_values
        final_df[f"best_distance_{suffix}"] = best_distance_values
        final_df[f"best_frequency_{suffix}"] = best_frequency_values
        final_df[f"all_suggestions_{suffix}"] = suggestion_values

meta_df = pd.DataFrame(
    [
        {"parameter": "edit_distances", "value": ", ".join(map(str, EDIT_DISTANCES))},
        {"parameter": "lambda_values", "value": ", ".join(map(str, LAMBDA_VALUES))},
        {"parameter": "prefix_length", "value": PREFIX_LENGTH},
        {"parameter": "min_word_frequency", "value": MIN_WORD_FREQUENCY},
        {"parameter": "dictionary_size", "value": len(dictionary_terms)},
        {"parameter": "input_rows", "value": len(base_df)},
        {"parameter": "abbreviation_paths", "value": f"{abbreviation_path.name}, {attention_path.name}"},
    ]
)

output_path = base_dir / "txtde_olmayan_kelimeler_Kontrol_duzeltilmis_all_lambdas.xlsx"
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    final_df.to_excel(writer, sheet_name="sonuc", index=False)
    meta_df.to_excel(writer, sheet_name="parametreler", index=False)

print("\nKaydedildi:")
print(output_path)
print("\nİlk 20 sonuç sütunları:\n")
preview_columns = [source_column]
for edit_distance in EDIT_DISTANCES:
    for lambda_value in LAMBDA_VALUES:
        suffix = combo_suffix(edit_distance, lambda_value)
        preview_columns.extend([f"corrected_{suffix}", f"confidence_{suffix}"])
print(final_df.head(20)[preview_columns])